# Data Scientist (DS) — Coordenador de Aprendizado Federado (`syft-flwr` + Google Drive)

Este notebook representa o **Data Scientist (DS)**, responsável por se conectar aos **3 Data Owners (DO1, DO2 e DO3)**, cada um com **1/3** do conjunto de dados *Pima Indians Diabetes*, explorar os datasets *mock*, submeter o projeto Flower e executar o agregador federado.

### Pré-verificação Obrigatória (Aba Lateral do Colab)
Antes de começar (ou ao repetir o experimento do zero):
1. Clique no ícone de **Arquivos** (pasta) na aba lateral esquerda do Colab (`/content`).
2. Confirme que **não existem** pastas de execuções anteriores (`SyftBox_*`, `dataset/` ou `outputs/`).
3. Se estiver repetindo o experimento após um erro ou rodada anterior, rode primeiro `do_client.delete_syftbox()` (Etapa 9) e depois clique em **Ambiente de execução → Desconectar e excluir ambiente de execução** para começar com o disco e o Google Drive limpos.

### Ordem de Execução (Grupo: DO1, DO2, DO3 e DS)
> **Atenção:** Cada integrante do grupo deve usar **uma conta Google diferente**.

1. **ETAPA 1 (DO1, DO2, DO3 e DS):** Instalar `syft-flwr` e fazer login com **4 contas Google distintas**.
2. **ETAPA 2 (DS):** Adicionar DO1, DO2 e DO3 como *peers* (`add_peer`).
3. **ETAPA 3 (DO1, DO2 e DO3):** Carregar *peers* e **aprovar o DS** (`approve_peer_request`) até o estado mudar para `ACCEPTED`.
4. **ETAPA 4 (DO1, DO2 e DO3):** Baixar os dados da URL *raw* do GitHub, gerar a partição `2` (1/3 dos dados), **inspecionar seus dados privados vs. mock**, criar o dataset e rodar `sync()`.
5. **ETAPA 5 (DS):** Aguardar ~15s, rodar `ds_client.sync()` e validar os datasets dos 3 DOs.
6. **ETAPA 6 (DS):** Baixar o projeto Flower (do repositório próprio), fazer `bootstrap` e submeter os jobs para DO1, DO2 e DO3.
7. **ETAPA 7 (DO1, DO2 e DO3):** Sincronizar, aprovar o job recebido e executar `process_approved_jobs()`.
8. **ETAPA 8 (DS):** Executar o servidor agregador Flower (`main.py`).
9. **ETAPA 9 (Todos — Opcional):** Limpar os recursos do SyftBox (`delete_syftbox()`).

---
## ETAPA 1 — Instalação e Login do Data Scientist (DS)
*(Execute em paralelo com DO1, DO2 e DO3)*

Use uma **conta Google exclusiva** para o DS (diferente das contas usadas em DO1, DO2 e DO3).

In [ ]:
# Instalação recomendada via PyPI com versões específicas
!uv pip install -q "syft-flwr==0.5.3" "syft-client==0.1.94"

# ==============================================================================
# [PLANO B - RESERVA 1] Instalar de um fork próprio travado em um commit específico:
# !uv pip install -q "git+https://github.com/erickjun321/syft-flwr.git@<HASH_DO_COMMIT>"
#
# [PLANO B - RESERVA 2] Instalar direto do branch main oficial (use apenas se o PyPI falhar):
# !uv pip install -q "git+https://github.com/OpenMined/syft-flwr.git@main"
# ==============================================================================

In [ ]:
import syft_client as sc
import syft_flwr

print(f"syft_client = {sc.__version__}")
print(f"syft_flwr   = {syft_flwr.__version__}")

ds_email = input("Digite o e-mail Google do Data Scientist (DS): ").strip()
ds_client = sc.login_ds(email=ds_email)

---
## ETAPA 2 — Adicionar DO1, DO2 e DO3 como Peers
*(Execute após a Etapa 1 estar concluída em todos os notebooks)*

> **Dica:** Certifique-se de digitar corretamente os e-mails Gmail usados no login de cada DO.

In [ ]:
do1_email = input("Digite o e-mail Google do DO1: ").strip()
do2_email = input("Digite o e-mail Google do DO2: ").strip()
do3_email = input("Digite o e-mail Google do DO3: ").strip()

do_emails_list = [do1_email, do2_email, do3_email]

for email in do_emails_list:
    ds_client.add_peer(email)
    print(f"Pedido de peer enviado para: {email}")

print("Peers cadastrados no DS:", ds_client.peers)

---
## PAUSA — Aguarde DO1, DO2 e DO3 concluírem as ETAPAS 3 e 4

Antes de prosseguir para a Etapa 5, confirme que **todos os 3 DOs**:
1. Executaram a **Etapa 3** (`approve_peer_request`) e exibem `PeerState.ACCEPTED` para o DS.
2. Executaram a **Etapa 4** (`create_dataset(..., sync=True)` + `do_client.sync()`).
3. Aguarde cerca de **15 segundos** após o último `sync()` para o Google Drive propagar as permissões.

---
## ETAPA 5 — Sincronizar e Explorar os Datasets dos 3 DOs 
*(Execute após a Etapa 4 de DO1, DO2 e DO3)*

In [ ]:
# Sincronização explícita
ds_client.sync()

for nome, email in [("DO1", do1_email), ("DO2", do2_email), ("DO3", do3_email)]:
    datasets = ds_client.datasets.get_all(datasite=email)
    print(f"=== {nome} ({email}) — {len(datasets)} dataset(s) encontrado(s) ===")
    if not datasets:
        print("  [AVISO] Lista vazia! Verifique se este DO aprovou o DS (ACCEPTED) antes de criar o dataset e rode sync().")
    else:
        print(datasets[0].describe())

---
## ETAPA 6 — Baixar Projeto Flower, Fazer Bootstrap e Submeter Jobs aos 3 DOs
*(Execute somente após confirmar que DO1, DO2 e DO3 retornaram o dataset na Etapa 5)*

- **Fonte principal:** Arquivo estático `fl-diabetes-prediction.tar.gz`.
- **Planos B (Reserva):** Estão comentados abaixo caso prefira fazer upload manual do `.tar.gz` no Colab ou baixar do repositório original.

In [ ]:
from pathlib import Path
import subprocess

SYFT_FLWR_PROJECT_PATH = Path("/content/fl-diabetes-prediction")
LOCAL_TAR_PATH = Path("/content/fl-diabetes-prediction.tar.gz")
PROJECT_TAR_URL = "https://raw.githubusercontent.com/erickjun321/Aprendizado-Federado/refs/heads/main/fl-diabetes-prediction.tar.gz"

if not SYFT_FLWR_PROJECT_PATH.exists():
    SYFT_FLWR_PROJECT_PATH.mkdir(parents=True, exist_ok=True)

    # Baixa o .tar.gz estático do repositório próprio (se ainda não estiver em /content)
    if not LOCAL_TAR_PATH.exists():
        subprocess.run(
            ["curl", "-fSL", "--retry", "3", "--retry-delay", "2", PROJECT_TAR_URL, "-o", str(LOCAL_TAR_PATH)],
            check=True,
        )

    # Extrai o conteúdo para /content/fl-diabetes-prediction
    subprocess.run(
        ["tar", "-xzf", str(LOCAL_TAR_PATH), "-C", str(SYFT_FLWR_PROJECT_PATH)],
        check=True,
    )
    print(f"Projeto syft-flwr extraído com sucesso em: {SYFT_FLWR_PROJECT_PATH}")
else:
    print(f"Usando cópia local já existente em: {SYFT_FLWR_PROJECT_PATH}")

# ==============================================================================
# [PLANO B - RESERVA 1] Upload manual do fl-diabetes-prediction.tar.gz:
# 1) Arraste 'fl-diabetes-prediction.tar.gz' para /content na aba lateral
#    (ou descomente as 2 linhas abaixo) e reexecute esta célula:
# from google.colab import files
# uploaded = files.upload()
#
# [PLANO B - RESERVA 2] Download direto do repositório original (com retentativa):
# !rm -rf /content/fl-diabetes-prediction
# !mkdir -p /content/fl-diabetes-prediction
# !curl -fSL --retry 3 --retry-delay 2 https://github.com/khoaguin/fl-diabetes-prediction/archive/refs/heads/main.tar.gz | tar -xz --strip-components=1 -C /content/fl-diabetes-prediction
# ==============================================================================

# Validação rápida da estrutura extraída
assert (SYFT_FLWR_PROJECT_PATH / "pyproject.toml").exists(), (
    "pyproject.toml não encontrado em /content/fl-diabetes-prediction. "
    "Verifique se o .tar.gz foi compactado sem pasta raiz extra (ou use --strip-components=1)."
)
print("Estrutura do projeto validada (pyproject.toml encontrado).")

In [ ]:
import syft_flwr

try:
    !rm -rf {SYFT_FLWR_PROJECT_PATH / "main.py"}
    print(f"syft_flwr version = {syft_flwr.__version__}")
    do_emails = [peer.email for peer in ds_client.peers]
    print(f"Datasites participantes ({len(do_emails)}): {do_emails}")

    syft_flwr.bootstrap(
        SYFT_FLWR_PROJECT_PATH,
        aggregator=ds_email,
        datasites=do_emails,
    )
    print("Projeto configurado (bootstrap) com sucesso!")
except Exception as e:
    print("Erro no bootstrap:", e)

In [ ]:
!rm -rf {SYFT_FLWR_PROJECT_PATH / "fl_diabetes_prediction" / "__pycache__"}

job_name = "fl-diabetes-training"

for email in [do1_email, do2_email, do3_email]:
    ds_client.submit_python_job(
        user=email,
        code_path=str(SYFT_FLWR_PROJECT_PATH),
        job_name=job_name,
    )
    print(f"Job '{job_name}' submetido para {email}")

ds_client.sync()
ds_client.jobs

---
## PAUSA — Aguarde DO1, DO2 e DO3 executarem a ETAPA 7

Confirme que **DO1, DO2 e DO3** já aprovaram o job e iniciaram `do_client.process_approved_jobs()` antes de iniciar o agregador abaixo.

---
## ETAPA 8 — Instalar Dependências e Executar o Agregador Federado (Flower)

In [ ]:
!uv pip install -q \
    "flwr-datasets>=0.5.0" \
    "imblearn>=0.0" \
    "loguru>=0.7.3" \
    "pandas>=2.3.0" \
    "ipywidgets>=8.1.7" \
    "scikit-learn==1.7.1" \
    "torch>=2.8.0" \
    "ray==2.31.0" 

In [ ]:
assert SYFT_FLWR_PROJECT_PATH.exists(), f"Caminho do projeto não encontrado: {SYFT_FLWR_PROJECT_PATH}"
assert (SYFT_FLWR_PROJECT_PATH / "main.py").exists(), f"main.py não encontrado em: {SYFT_FLWR_PROJECT_PATH / 'main.py'}"

ds_email = ds_client.email
syftbox_folder = f"/content/SyftBox_{ds_email}"

!SYFTBOX_EMAIL="{ds_email}" SYFTBOX_FOLDER="{syftbox_folder}" uv run {str(SYFT_FLWR_PROJECT_PATH / "main.py")}

In [ ]:
# Verifica o status final dos jobs nos 3 DOs após o treinamento
ds_client.sync()
ds_client.jobs

---
## ETAPA 9 — Limpeza (Opcional)
*(Execute somente quando quiser remover todos os recursos do SyftBox do DS)*

> **Para repetir a experiência do zero:** rode `ds_client.delete_syftbox()` abaixo e, em seguida, clique em **Ambiente de execução → Desconectar e excluir ambiente de execução**.

In [ ]:
# Descomente a linha abaixo apenas quando quiser apagar o SyftBox do DS (local + Google Drive)
# ds_client.delete_syftbox()